In [ ]:
import cv2
import time
import subprocess
import random
import numpy as np
from gtts import gTTS
import pygame

pygame.mixer.init()

face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

phone_threshold_seconds = 900

insults_phone = [
#here i add my insults
]

insults_gesture = [

#here i add my insults

]

def speak(text):
    print(f"\n[Insult Bot]: {text}\n")
    try:
        tts = gTTS(text=text, lang='en')
        tts.save("temp_speech.mp3")
        pygame.mixer.music.load("temp_speech.mp3")
        pygame.mixer.music.play()
        while pygame.mixer.music.get_busy():
            time.sleep(0.1)
        pygame.mixer.music.unload()
    except Exception as e:
        print(f"Audio error: {e}")

def has_upcoming_deadlines():
    try:
        result = subprocess.run(["calcurse", "-r5"], capture_output=True, text=True)
        return len(result.stdout.strip()) > 0
    except Exception as e:
        print(f"Calendar check error: {e}")
        return False

cap = cv2.VideoCapture(0)

prev_gray = None
gesture_cooldown = 0
phone_timer_start = None


try:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        current_time = time.time()
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        if prev_gray is not None:
            flow = cv2.calcOpticalFlowFarneback(
                prev_gray, gray, None, 0.5, 3, 15, 3, 5, 1.2, 0
            )
            v_motion = flow[..., 1]
            h, w = gray.shape
            left_half_v = v_motion[:, :w // 2]
            right_half_v = v_motion[:, w // 2:]

            avg_left = np.mean(left_half_v)
            avg_right = np.mean(right_half_v)

            is_opposite_motion = (avg_left * avg_right < -0.2) and (abs(avg_left) > 0.5 or abs(avg_right) > 0.5)

            if is_opposite_motion:
                if current_time - gesture_cooldown > 5:
                    speak(random.choice(INSULTS_GESTURE))
                    gesture_cooldown = current_time

        prev_gray = gray

        blurred = cv2.GaussianBlur(gray, (5, 5), 0)
        _, thresh = cv2.threshold(blurred, 60, 255, cv2.THRESH_BINARY_INV)
        contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        phone_detected = False
        for cnt in contours:
            area = cv2.contourArea(cnt)
            if 5000 < area < 40000:
                x, y, w_rect, h_rect = cv2.boundingRect(cnt)
                aspect_ratio = float(h_rect) / w_rect
                if 1.4 < aspect_ratio < 2.5 or 0.4 < aspect_ratio < 0.7:
                    phone_detected = True
                    cv2.rectangle(frame, (x, y), (x + w_rect, y + h_rect), (0, 0, 255), 2)
                    break

        if phone_detected:
            if phone_timer_start is None:
                phone_timer_start = current_time
            elif current_time - phone_timer_start > PHONE_THRESHOLD_SECONDS:
                if has_upcoming_deadlines():
                    speak(random.choice(INSULTS_PHONE))
                phone_timer_start = current_time
        else:
            phone_timer_start = None

        cv2.imshow('Insult Bot - Vision Feed', frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

finally:
    cap.release()
    cv2.destroyAllWindows()